# Task 05 — Remove outliers from training

## 🌌 Macrocosm photo-z — outlier study (tasks 2026-06-16)

**✅ GIVEN (do NOT re-derive this):** we already ran a **400k 5-fold cross-validation** with three
models (HGB, RF, MLP) and collected the galaxies that *all three* predict catastrophically out-of-fold
(|Δz/(1+z)| > 0.05). There are **6,974** such **hard** galaxies. Their objids are in
**`../hard_objids.csv`**. Treat this hard set as a **fixed input** — your job is to characterize / act
on it, not to re-find it from 400k.

**Catalog:** `gs://macrocosm-lewagon/data/sample_v1/catalog_v1.parquet` (600k rows, 55 columns).
The SDSS **`-9999` sentinel** means "not measured" — always clean it to NaN first.

**Metric:** `σ_MAD = 1.4826 · median(|Δz − median(Δz)|)`, with `Δz = (z_pred − z_true)/(1+z)`;
an **outlier** is `|Δz| > 0.05`. σ_MAD is the headline metric, report **outlier rate** alongside it.

---

Test whether the hard galaxies *pollute* training. Remove them from the training folds (keep the test
fold intact) and see if the remaining (normal) galaxies improve — and whether that depends on the model.

## ⚠️ Dependency

This task builds on **Task 02 — baseline models**. Before you start, make sure the prerequisite is **completed — by you or a teammate — and merged into `2026.6.16`**. If it isn't ready yet, coordinate with whoever owns it to finish the prerequisite first; or, of course, just wait until it lands.

In [6]:
# === shared setup: load catalog, clean -9999, build the 16 features ===
import pandas as pd, numpy as np

CATALOG = "/Users/cathyzhou/code/Le-Wagon-Macrocosm/Macrocosm/catalog_v1.parquet"
# Colab: from google.colab import auth; auth.authenticate_user()
# or download once: !gcloud storage cp $CATALOG . , then CATALOG = "catalog_v1.parquet"

FEATS = ["dered_u","dered_g","dered_r","dered_i","dered_z","g-r","u-g","r-i","i-z",
         "log_expRad_r","log_deVRad_r","log_petroRad_r","log_petroR50_r","log_petroR90_r",
         "fracDeV_r","conc_r"]

def load_features(path=CATALOG, n=None, seed=0):
    """Load catalog, clean the -9999 sentinel, build colors / log-sizes / conc.
    Returns (D, cat): D = features+redshift+objid (optionally subsampled), cat = full cleaned catalog."""
    cat = pd.read_parquet(path)
    num = cat.select_dtypes("number").columns
    cat[num] = cat[num].mask(cat[num] <= -100)                       # clean SDSS -9999
    for a, b in [("u","g"),("g","r"),("r","i"),("i","z")]:
        cat[f"{a}-{b}"] = (cat[f"dered_{a}"] - cat[f"dered_{b}"]).clip(-1, 4)
    for s in ["expRad_r","deVRad_r","petroRad_r","petroR50_r","petroR90_r"]:
        cat["log_"+s] = np.log1p(cat[s].clip(lower=0))
    cat["conc_r"] = cat["petroR90_r"] / cat["petroR50_r"].replace(0, np.nan)
    D = cat[FEATS + ["redshift","objid"]].replace([np.inf,-np.inf], np.nan).dropna()
    if n:
        D = D.sample(n, random_state=seed).reset_index(drop=True)
    return D, cat

def smad(dz): return 1.4826 * np.median(np.abs(dz - np.median(dz)))

def metrics(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    dz = (y_pred - y_true) / (1 + y_true)
    return {"MAE": round(float(np.mean(np.abs(y_pred-y_true))), 5),
            "sigma_MAD": round(float(smad(dz)), 5),
            "outlier_rate": round(float(np.mean(np.abs(dz) > 0.05)), 5)}

# the GIVEN hard set (6,974 objids from the 400k 5-fold CV)
HARD = set(pd.read_csv("../hard_objids.csv")["objid"])
print(f"hard set: {len(HARD)} galaxies")

hard set: 6974 galaxies


👇 Reuse the `models()` from Task 02 (RF, HGB, MLP — drop LR here).

❓ **Question (two regimes)** ❓

👇 100k (`seed=1`), `KFold(3, ...,42)`. For each model, get OOF predictions **(a) trained normally** and **(b) trained with the hard objids removed from each training fold** (test fold unchanged).

In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

D, cat = load_features(n=100000, seed=1)
is_hard = D['objid'].isin(HARD).values

❓ **Question (normal vs hard)** ❓

👇 Split the test metrics into the **normal** subset and the **hard** subset. Does removing outliers help the normal galaxies? Compare Δσ_MAD across RF / HGB / MLP.

In [9]:
# YOUR CODE HERE
sub = cat[cat['objid'].isin(D['objid'])].copy()
sub['hard'] = sub['objid'].isin(HARD)
sub['normal'] = ~sub['hard']

print(f"hard: {sub['hard'].sum()}  normal: {sub['normal'].sum()}")
#print(f"subsample: {len(sub):,}  hard: {sub['hard'].sum()}  normal: {(~sub['hard']).sum()}")

hard: 1144  normal: 98856


In [ ]:
sub['normal']

9         True
13        True
14        True
15        True
46        True
          ... 
599967    True
599969    True
599971    True
599990    True
599992    True
Name: normal, Length: 100000, dtype: bool

In [18]:
models = {"RF":  RandomForestRegressor(n_estimators=150, min_samples_leaf=2, n_jobs=-1, random_state=0),
        "HGB": HistGradientBoostingRegressor(max_iter=300, learning_rate=0.1, early_stopping=True, random_state=0),
        "MLP": make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(128,64,32), alpha=1e-4,
                   batch_size=1024, learning_rate_init=1e-3, early_stopping=True, n_iter_no_change=12,
                   max_iter=100, random_state=0))}

In [31]:
y = D.loc[~D['objid'].isin(HARD), "redshift"]
y

0        0.071959
1        0.094197
2        0.223124
3        0.132936
4        0.077613
           ...   
99995    0.140045
99996    0.103615
99997    0.309235
99998    0.282521
99999    0.061692
Name: redshift, Length: 98856, dtype: float64

In [32]:
from sklearn.base import clone

# X needs to drop values where it is hard
X= D.loc[~D['objid'].isin(HARD), FEATS]
y = D.loc[~D['objid'].isin(HARD), "redshift"]

kf = KFold(3, shuffle=True, random_state=42)

results = {}

for model_name, model in models.items():
    oof_predictions = np.full(len(y), np.nan)

    for train_idx, val_idx in kf.split(X):
        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_train = y.iloc[train_idx]

        fitted_model = clone(model)
        fitted_model.fit(X_train, y_train)

        oof_predictions[val_idx] = fitted_model.predict(X_val)

    results[model_name] = metrics(y, oof_predictions)

results_df = pd.DataFrame(results).T
print('model trained on easy to read galaxies ', results_df)

model trained on easy to read galaxies           MAE  sigma_MAD  outlier_rate
RF   0.01530    0.01403       0.01992
HGB  0.01534    0.01425       0.01913
MLP  0.01534    0.01404       0.02073


In [ ]:
is_hard = D['objid'].isin(HARD).values

X = D[FEATS]
ids = D['objid']
y = D['redshift']

for model_name, model in models.items():
    oof_predictions = np.full(len(y), np.nan)

    for train_idx, val_idx in kf.split(X):
        train_mask = ~is_hard[train_idx]
        X_train = X.iloc[train_idx][train_mask]
        y_train = y[train_idx][train_mask]

        X_val = X.iloc[val_idx]

        fitted_model = clone(model)
        fitted_model.fit(X_train, y_train)

        oof_predictions[val_idx] = fitted_model.predict(X_val)

    results[model_name] = metrics(y, oof_predictions)

results_df = pd.DataFrame(results).T
print('model trained on easy to read galaxies, validation untouched', results_df)


/Users/cathyzhou/.pyenv/versions/3.10.6/envs/macrocosm/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


model trained on easy to read galaxies, validation untouched          MAE  sigma_MAD  outlier_rate
RF   0.01623    0.01423       0.02988
HGB  0.01622    0.01437       0.02910
MLP  0.01629    0.01432       0.03016


In [41]:
results_df['MAE']['RF']

np.float64(0.01623)

In [43]:
from sklearn.base import clone

D, cat = load_features(n=100000, seed=1)
X, y = D[FEATS], D["redshift"]

kf = KFold(3, shuffle=True, random_state=42)

results = {}

for model_name, model in models.items():
    oof_predictions = np.full(len(y), np.nan)

    for train_idx, val_idx in kf.split(X):
        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_train = y.iloc[train_idx]

        fitted_model = clone(model)
        fitted_model.fit(X_train, y_train)

        oof_predictions[val_idx] = fitted_model.predict(X_val)

    results[model_name] = metrics(y, oof_predictions)

results_df_all = pd.DataFrame(results).T
print(results_df_all)

         MAE  sigma_MAD  outlier_rate
RF   0.01633    0.01432       0.03059
HGB  0.01640    0.01460       0.03023
MLP  0.01670    0.01487       0.03240


In [44]:
hard_effect = results_df_all/results_df
hard_effect

,MAE,sigma_MAD,outlier_rate
RF,1.006161,1.006325,1.023762
HGB,1.011097,1.016006,1.038832
MLP,1.025169,1.038408,1.074271


## 📝 Write your report

In [45]:
# === write_report: run this after you've filled in your results, it generates report.md ===
def write_report(title, results: dict, conclusion: str, path="report.md"):
    lines = [f"# {title}", "", "_Auto-generated by task.ipynb — fill `results` and `conclusion` above._", "",
             "## Results", ""]
    for k, v in results.items():
        lines.append(f"- **{k}**: {v}")
    lines += ["", "## Conclusion", "", conclusion, ""]
    with open(path, "w") as f:
        f.write("\n".join(lines))
    print("wrote", path)

In [46]:
write_report("Task 05 — Remove outliers from training",
             {"normal sigma_MAD keep->remove per model": results_df['sigma_MAD'], "hard subset effect": hard_effect},
             "All models benefit from dropping the hard subset, but random forest benefits the least")

wrote report.md


In [47]:
# === Commit & push your results (run last; make sure you are on branch 2026.6.16) ===
# First time: git pull origin 2026.6.16   to get the latest.
!git add task.ipynb report.md && git commit -m "05-remove-outliers-training task" && git push origin 2026.6.16

[2026.6.16 4a85beb] 05-remove-outliers-training task
 1 file changed, 16 insertions(+), 1 deletion(-)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 8 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 814 bytes | 814.00 KiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To github.com:Le-Wagon-Macrocosm/Macrocosm.git
   4a9adc0..4a85beb  2026.6.16 -> 2026.6.16


## 🔭 Go further (optional)

Play around with the data and the results you now have. If you find anything new — an unexpected pattern, a useful feature, a failure mode we missed — write it up and post it to our **YouTrack Knowledge Base** so the whole team benefits.